# Adım 4 — Anahtar kelime filtresi (referans yöntem)

**Bu defter ne yapar?** Hiç yapay zeka eğitmeden, elle yazdığımız **bir kelime/kalıp listesine** bakan çok basit bir "güvenlik görevlisi" kurar. Metinde "ignore previous instructions" gibi tipik saldırı kalıplarından biri geçiyorsa **zararlı** der, geçmiyorsa **zararsız** der.

**Neden bunu yapıyoruz?** Özette söz verdik: BERT/RoBERTa'yı basit bir filtreyle karşılaştıracağız. Basit filtre bile iyi çıkabilir (literatürde böyle örnekler var). Eğer BERT/RoBERTa bunu açık farkla geçemezse, "yapay zeka şart mı?" sorusunun cevabı da bildiride yer alır. Bu yüzden önemli bir referans.

**Dürüstlük kuralı:** Aşağıdaki kalıp listesi **sonuçlara bakılarak değiştirilmeyecek.** Liste genel bilgiyle önceden yazıldı. Sonuçları gördükten sonra listeyi iyileştirirsek bunu bildiride açıkça yazmak zorundayız.
(Not: Almanca birkaç kalıp var, çünkü `deepset` veri setinde Almanca örnekler olduğunu 01. adımda gördük.)

**Çalıştırma:** `Çalışma zamanı → Tümünü çalıştır`. Bilgisayarınızdaki `veri_birlesik.csv` dosyasını (03. adımdan inen) seçmeniz istenebilir.

## 1) Kütüphaneler ve veri dosyası
Dosyayı önce Drive'da arar. Bulamazsa **"Dosya seç"** düğmesi çıkar: İndirilenler klasöründeki `veri_birlesik.csv` dosyasını seçin.

In [ ]:
import re, glob, os
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def csv_yukle():
    # 1) Colab'da bu klasörde zaten varsa
    for yol in ["veri_birlesik.csv"] + glob.glob("/content/drive/MyDrive/**/veri_birlesik.csv", recursive=True):
        if os.path.exists(yol):
            print("Bulundu:", yol)
            return pd.read_csv(yol)
    # 2) Bulunamadıysa bilgisayardan seçtir
    from google.colab import files
    yuklenen = files.upload()
    ad = list(yuklenen.keys())[0]
    return pd.read_csv(ad)

df = csv_yukle()
print(df.shape)
print(df.groupby(["rol", "kaynak", "bolum"]).label.agg(satir="size", zararli="sum"))

## 2) Saldırı "ailesi" sütunu
`wambosec` veri setinde saldırı türü adları dağınıktı (aynı tür 5 farklı yazımla geçiyordu: `Tier 1 - Semantic Camouflage`, `semantic camouflage`, `semantic-camouflage`...). Bunları **14 geniş aileye** topluyoruz (kodlama, steganografi, unicode/homoglif, vb.).
`neuralchemy` kategorileri zaten temiz, olduğu gibi `aile` olur. Zararsız örneklerin ailesi `yok`.
Bu aile sütunu sonra "hangi saldırı türünü kaçırıyoruz?" analizi ve "bir saldırı ailesini eğitimden çıkar" deneyi için kullanılacak.

In [ ]:
def kategori_temizle(k):
    k = str(k).lower().strip()
    k = re.sub(r"^tier\s*\d+\s*-\s*", "", k)          # "Tier 3 - " önekini at
    return re.sub(r"[\s_/\-&]+", " ", k).strip()         # ayraçları tek boşluk yap

# Sıra önemli: ilk uyan aile seçilir
AILE_KURALLARI = [
    ("steganografi",         ["stegan", "whitespace", "first letter", "acrostic", "zero width"]),
    ("unicode_homoglif",     ["unicode", "homoglyph", "fullwidth", "rtl", "combining", "confusable", "bidi", "braille", "zalgo"]),
    ("ascii_art",            ["ascii art"]),
    ("kodlama",              ["encoding", "base64", "hex", "rot13", "url", "html entit", "punycode", "nested", "binary", "leetspeak"]),
    ("bozuk_sozdizimi",      ["malformed", "broken", "unclosed", "incomplete"]),
    ("bicim_dizgisi",        ["format string", "format ansi", "ansi"]),
    ("dil_karisimi",         ["language", "romanized"]),
    ("dilbilimsel_hile",     ["linguistic"]),
    ("anlamsal_gizleme",     ["semantic"]),
    ("baglam_manipulasyonu", ["context"]),
    ("jailbreak_kaliplari",  ["jailbreak"]),
    ("sosyal_muhendislik",   ["social"]),
    ("yetki_taklidi",        ["authority"]),
    ("teknik_gizleme",       ["technical", "obfuscation"]),
]

def aile_ata(satir):
    if satir["label"] == 0:
        return "yok"
    k = satir["kategori"]
    if pd.isna(k):
        return "bilinmiyor"                      # safeguard/jackhhao/deepset: kategori sütunu yok
    if satir["kaynak"] == "wambosec":
        n = kategori_temizle(k)
        for ad, anahtarlar in AILE_KURALLARI:
            if any(a in n for a in anahtarlar):
                return ad
        return "diger"
    return str(k)                                # neuralchemy: olduğu gibi

df["aile"] = df.apply(aile_ata, axis=1)
print("wambosec aileleri (zararlı örnek sayısı):")
print(df[(df.kaynak == "wambosec") & (df.label == 1)].aile.value_counts().to_string())

## 3) Kalıp listesi (anahtar kelime filtresi)
Her satır bir **düzenli ifade** (regex = esnek arama kalıbı). Örneğin `ignore (all )?previous instructions` şunları yakalar: "ignore previous instructions", "ignore all previous instructions".
Metin önce küçük harfe çevrilir. **Skor = metinde kaç farklı kalıp bulunduğu.** Skor 1 veya daha fazlaysa "zararlı" deriz.

In [ ]:
KALIPLAR = [
    # "önceki talimatları yok say" ailesi
    r"ignore (all |any |the |your )?(previous|prior|above|earlier|preceding|former) (instructions|prompts?|rules|directions|messages|commands)",
    r"disregard (all |any |the |your )?(previous |prior |above |earlier )?(instructions|prompts?|rules|guidelines|directions)",
    r"forget (all |everything |your |the )?(previous |prior |above )?(instructions|rules|guidelines|training|prompts?)",
    r"new instructions?:",
    # sistem komutunu / gizli talimatı sızdırma
    r"system prompt",
    r"(reveal|show|print|repeat|display|leak|output) (me )?(your|the) (system |initial |original |hidden |secret )?(prompt|instructions)",
    # rol değiştirme / jailbreak
    r"you are now ",
    r"from now on,? you",
    r"pretend (to be|you are|that you)",
    r"\bact as (a |an |if )",
    r"developer mode",
    r"\bdan\b",
    r"do anything now",
    r"jailbr(eak|oken)",
    r"(stay|remain) in character",
    r"(never|do not|don't) (refuse|decline)",
    # güvenlik önlemlerini kapatma
    r"(without|no) (any )?(restrictions|limitations|filters|censorship|ethical|moral)",
    r"(disable|turn off|remove|bypass) (your |all |the )?(safety|filters?|guardrails?|restrictions|content polic)",
    r"\b(unfiltered|uncensored)\b",
    # Almanca birkaç tipik kalıp (deepset'te Almanca örnek var)
    r"ignoriere? (alle |die )?(vorherigen|obigen|bisherigen)",
    r"vergiss (alles|alle|deine)",
    r"anweisungen (ignorieren|vergessen)",
]
DERLENMIS = [re.compile(k) for k in KALIPLAR]

def skor_hesapla(metin):
    m = re.sub(r"\s+", " ", str(metin).lower())
    return sum(1 for k in DERLENMIS if k.search(m))

df["skor"] = df.text.map(skor_hesapla)
df["tahmin"] = (df.skor >= 1).astype(int)
print("Kalıp sayısı:", len(KALIPLAR))
print("Skor dağılımı:")
print(df.skor.value_counts().sort_index().to_string())

## 4) Ölçüm
Hangi veri parçalarını ölçüyoruz? Filtre hiç eğitilmediği için **her veri setinin tamamını** ölçebiliriz. BERT/RoBERTa ile adil kıyas için BERT tarafında kullanılacak parçalarla aynı tanımı kullanıyoruz:
- `neuralchemy`, `safeguard`, `wambosec`: hem **tamamı** hem sadece **test bölümü**
- `jackhhao`: sadece **test bölümü** (255 satır). Eğitim bölümünde neredeyse hiç zararlı örnek kalmadı, o yüzden kullanmıyoruz.
- `deepset`: **tamamı** (656 satır). Küçük ve çok dilli, **ek bilgi** olarak okunmalı.

Ölçütler: doğruluk, kesinlik (zararlı dediklerimizin ne kadarı gerçekten zararlı), duyarlılık (zararlıların ne kadarını yakaladık), F1, **yanlış pozitif oranı** (masumların yüzde kaçını boş yere engelledik), ROC-AUC (skor olarak kalıp sayısını kullanıyoruz).

In [ ]:
def olc(d):
    y, p, s = d.label.values, d.tahmin.values, d.skor.values
    tn = int(((y == 0) & (p == 0)).sum()); fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum()); tp = int(((y == 1) & (p == 1)).sum())
    return dict(
        satir=len(d), zararli=int(y.sum()),
        dogruluk=round(accuracy_score(y, p), 4),
        kesinlik=round(precision_score(y, p, zero_division=0), 4),
        duyarlilik=round(recall_score(y, p, zero_division=0), 4),
        f1=round(f1_score(y, p, zero_division=0), 4),
        yanlis_pozitif_orani=round(fp / (fp + tn), 4) if (fp + tn) else None,
        roc_auc=round(roc_auc_score(y, s), 4) if len(set(y)) == 2 else None,
        TP=tp, FN=fn, FP=fp, TN=tn)

parcalar = {}
for ad in ["neuralchemy", "safeguard", "wambosec"]:
    d = df[df.kaynak == ad]
    parcalar[f"{ad} (tamami)"] = d
    parcalar[f"{ad} (test)"] = d[d.bolum == "test"]
parcalar["jackhhao (test)"] = df[(df.kaynak == "jackhhao") & (df.bolum == "test")]
parcalar["deepset (tamami)"] = df[df.kaynak == "deepset"]

sonuc = pd.DataFrame({ad: olc(d) for ad, d in parcalar.items()}).T
sonuc[["satir", "zararli", "TP", "FN", "FP", "TN"]] = sonuc[["satir", "zararli", "TP", "FN", "FP", "TN"]].astype(int)
pd.set_option("display.width", 250)
print(sonuc.to_string())

## 5) Hangi saldırı ailesini yakalıyor, hangisini kaçırıyor?
Her aile için **duyarlılık** (o ailedeki zararlıların yüzde kaçı yakalandı). Bu tablo bildiride "basit filtre hangi taktiklere kör?" analizinin temeli. Düşük olan aileler, BERT/RoBERTa'nın fark yaratması beklenen yerler.

In [ ]:
z = df[(df.label == 1) & (~df.aile.isin(["bilinmiyor"]))]
tablo = (z.groupby(["kaynak", "aile"]).agg(zararli_sayisi=("label", "size"), yakalanan=("tahmin", "sum")))
tablo["duyarlilik"] = (tablo.yakalanan / tablo.zararli_sayisi).round(3)
for ad in ["neuralchemy", "wambosec"]:
    print("\n==", ad, "==")
    print(tablo.loc[ad].sort_values("duyarlilik").to_string())

## 6) Hata örnekleri (hata analizi için)
Filtrenin **boş yere engellediği masum** örnekler (yanlış pozitif) ve **kaçırdığı zararlı** örnekler (yanlış negatif). İlk 100 karakter gösteriliyor. Bildiride "neden hata yaptı?" yorumunun malzemesi.

In [ ]:
ana = df[df.rol == "ana"]
print("=== Yanlış pozitif: masum ama engellendi (ilk 12) ===")
for _, r in ana[(ana.label == 0) & (ana.tahmin == 1)].head(12).iterrows():
    print("-", r.kaynak, "|", r.text[:100].replace("\n", " "))
print()
print("=== Yanlış negatif: zararlı ama kaçtı (ilk 12) ===")
for _, r in ana[(ana.label == 1) & (ana.tahmin == 0)].sample(12, random_state=0).iterrows():
    print("-", r.kaynak, "|", r.aile, "|", r.text[:100].replace("\n", " "))

## 7) Kaydet
`anahtar_kelime_sonuclari.csv` bilgisayarınıza iner. Drive'daki YiitliProje klasörüne sürükleyin. Ayrıca ekrandaki **4. ve 5. bölüm tablolarının** ekran görüntüsünü Claude'a verin.

In [ ]:
sonuc.to_csv("anahtar_kelime_sonuclari.csv")
from google.colab import files
files.download("anahtar_kelime_sonuclari.csv")